<a href="https://colab.research.google.com/github/Tungtom2004/Analysis-classification/blob/feat%2Fgru/GRU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
import warnings
import re
import string
import random


from wordcloud import WordCloud
import matplotlib.pyplot as plt
from nltk.tokenize import RegexpTokenizer , TweetTokenizer
from nltk.stem import WordNetLemmatizer ,PorterStemmer
from nltk.corpus import stopwords
from collections import defaultdict
from collections import Counter
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split


nlp = spacy.load("en_core_web_sm")
warnings.filterwarnings('ignore')

In [2]:
import string
import re
import nltk.corpus
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [3]:
df_train = pd.read_csv('/content/train.csv')

In [4]:
df_valid = pd.read_csv('/content/train.csv')
df_test = pd.read_csv('/content/test.csv')

In [5]:
df_train

,label,text
0,Positive,look ... kinda clean !
1,Positive,make dream become real .
2,Irrelevant,cant see clearly would willingly get switch . ...
3,Irrelevant,tried play pubg back 'm still bad ca n't sgshs...
4,Positive,really enjoy way leo capture game .
...,...,...
66590,Neutral,"walker alive , operation greenstone complete ...."
66591,Positive,NaN
66592,Negative,@ verizonsupport beware & warning time crisis ...
66593,Negative,@ nba2k_myteam 2k smt wife coz 're frustrated


In [6]:
def text_cleaning(text):
    emoji_pattern = re.compile(
        "["
        "\U0001f600-\U0001f64f"
        "\U0001f300-\U0001f5ff"
        "\U0001f680-\U0001f6ff"
        "\U0001f1e0-\U0001f1ff"
        "\U00002702-\U000027b0" # Thêm 0
        "\U000024c2-\U0001f251" # Thêm 0
        "]+", flags=re.UNICODE
    )
    text = text.lower()
    punc = str.maketrans(string.punctuation, ' '*len(string.punctuation))
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'\n',' ',text)
    text = emoji_pattern.sub(r' ',text)
    return text.strip()

In [7]:
Stopwords = set(nltk.corpus.stopwords.words("english")) - set(["not"])
def text_processing(text):
    processed_text = []
    lemma = WordNetLemmatizer()
    tokens = nltk.word_tokenize(text)

    for word in tokens:
        if word not in Stopwords:
            processed_text.append(lemma.lemmatize(word))
    return (' '.join(processed_text))

In [8]:
df_train

,label,text
0,Positive,look ... kinda clean !
1,Positive,make dream become real .
2,Irrelevant,cant see clearly would willingly get switch . ...
3,Irrelevant,tried play pubg back 'm still bad ca n't sgshs...
4,Positive,really enjoy way leo capture game .
...,...,...
66590,Neutral,"walker alive , operation greenstone complete ...."
66591,Positive,NaN
66592,Negative,@ verizonsupport beware & warning time crisis ...
66593,Negative,@ nba2k_myteam 2k smt wife coz 're frustrated


In [9]:
df_train['text'].isna().sum()
df_test['text'].isna().sum()
df_valid['text'].isna().sum()

np.int64(1203)

In [10]:
df_test['text'].isna().sum()

np.int64(126)

In [11]:
df_valid['text'].isna().sum()

np.int64(1203)

In [12]:
df_train.dropna(inplace=True)
df_test.dropna(inplace=True)
df_valid.dropna(inplace=True)

In [13]:
df_train['text'] = df_train['text'].apply(lambda x:text_cleaning(x))
df_valid['text'] = df_valid['text'].apply(lambda x:text_cleaning(x))
df_test['text'] = df_test['text'].apply(lambda x:text_cleaning(x))

In [14]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [15]:
df_train['text'] = df_train['text'].apply(lambda x:text_processing(x))
df_valid['text'] = df_valid['text'].apply(lambda x:text_processing(x))
df_test['text'] = df_test['text'].apply(lambda x:text_processing(x))


In [16]:
df_train.head()

,label,text
0,Positive,look ... kinda clean !
1,Positive,make dream become real .
2,Irrelevant,cant see clearly would willingly get switch . ...
3,Irrelevant,tried play pubg back 'm still bad ca n't sgshs...
4,Positive,really enjoy way leo capture game .


In [17]:
from nltk.tokenize import word_tokenize
from collections import Counter

max_len = 50
sen2num = {'Negative': 0, 'Positive': 1, 'Neutral': 2, 'Irrelevant': 3}

text_train = df_train['text'].tolist()
sen_train = df_train['label'].map(sen2num).tolist()
text_val = df_valid['text'].tolist()
sen_val = df_valid['label'].map(sen2num).tolist()
text_test = df_test['text'].tolist()
sen_test = df_test['label'].map(sen2num).tolist()

tokens = []
for text in text_train:
  split_text = word_tokenize(text)
  tokens.extend(split_text)

counter = Counter(tokens)
most_common = counter.most_common(10000)
vocab = {}
vocab['<PAD>'] = 0
for i, (word, _) in enumerate(most_common, start = 1):
  vocab[word] = i

vocab['<OOV>'] = len(vocab)
print("Vocab size:", len(vocab))

def text2numseq(text, vocab):
  split_text = word_tokenize(text)
  return [vocab.get(t, vocab['<OOV>']) for t in split_text]

seq_train = [text2numseq(text, vocab) for text in text_train]
seq_val = [text2numseq(text, vocab) for text in text_val]
seq_test = [text2numseq(text, vocab) for text in text_test]

def padding(sequence, max_len):
  padded_seq = []
  for seq in sequence:
    if len(seq) < max_len:
      seq = seq + (max_len - len(seq)) * [vocab['<PAD>']]
    else:
      seq = seq[:max_len]
    padded_seq.append(seq)
  return np.array(padded_seq)


pad_train = padding(seq_train, max_len)
pad_val = padding(seq_val, max_len)
pad_test = padding(seq_test, max_len)
print(pad_val[98:100])

Vocab size: 10002
[[ 448  428   80  140   18 2263  164  834   80    2   28  591    2   27
   271 1833    1    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0]
 [ 135   37 1955    1    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0]]


In [18]:
from torch.utils.data import TensorDataset, DataLoader

In [19]:
torch_train = torch.LongTensor(pad_train)
torch_sen_train = torch.LongTensor(sen_train)
torch_val = torch.LongTensor(pad_val)
torch_sen_val = torch.LongTensor(sen_val)
torch_test = torch.LongTensor(pad_test)
torch_sen_test = torch.LongTensor(sen_test)

dataset_train = TensorDataset(torch_train, torch_sen_train)
dataset_val = TensorDataset(torch_val, torch_sen_val)
dataset_test = TensorDataset(torch_test, torch_sen_test)

train_loader = DataLoader(dataset_train, batch_size = 64, shuffle=True)
val_loader = DataLoader(dataset_val, batch_size = 64)
test_loader = DataLoader(dataset_test, batch_size = 64)

print("Length val dataloader:", len(val_loader))

Length val dataloader: 1022


In [20]:
vocab_size = len(vocab)
embedding_dim = 200
embedding_matrix = np.zeros((vocab_size, embedding_dim))

In [ ]:
from thinc.layers.tensorflowwrapper import forward
class BiLSTM(nn.Module):
  def __init__(self, vocab_size, embedding_matrix, embedding_dim, hidden_dim, classes):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
    self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
    self.dropout = nn.Dropout(p = 0.2)
    self.linear = nn.Sequential(
        nn.Linear(hidden_dim * 2, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, classes)
    )

  def forward(self, x):
    x = self.embedding(x)
    x, _ = self.lstm(x)
    x = self.dropout(x)
    x = x[:, -1]
    y = self.linear(x)
    return y

hidden_dim = 100
classes = 4
model = BiLSTM(vocab_size, embedding_matrix, embedding_dim, hidden_dim, classes)

In [30]:
class GRUModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.gru = nn.GRU(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )
        self.dropout = nn.Dropout(0.2)

        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, classes)
        )

    def forward(self, x):
        x = self.embedding(x)              # (batch, seq, embed)
        output, hidden = self.gru(x)       # output: (batch, seq, hidden*2)
        x = self.dropout(output)
        x = x[:, -1, :]                   # lấy timestep cuối
        y = self.fc(x)
        return y

In [32]:
hidden_dim = 100
classes = 4
model = GRUModel(vocab_size, embedding_dim, hidden_dim, classes)

In [27]:
import torch.optim as optim
import torch.nn.functional as F
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

In [33]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

loss_fun = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 20

train_accuracy = []
train_losses = []
val_accuracy = []
val_losses = []

for epoch in range(epochs):
    model.train()
    losses = 0.0
    correct = 0
    total = 0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        preds = model(x)
        loss = loss_fun(preds, y)
        loss.backward()
        optimizer.step()

        losses += loss.item()* x.size(0)
        # softmax_outputs = F.softmax(preds, dim=1)
        _, res = torch.max(preds, 1)
        correct += torch.sum(res == y).item()
        total += y.size(0)

    train_acc = correct / total
    train_loss = losses / total
    train_accuracy.append(train_acc)
    train_losses.append(train_loss)

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            y = y.to(device)
            preds = model(x)
            loss = loss_fun(preds, y)

            val_loss += loss.item() * x.size(0)
            _, res = torch.max(preds, 1)
            val_correct += torch.sum(res == y).item()
            val_total += y.size(0)
            all_preds.extend(res.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    val_loss = val_loss / val_total
    val_acc = val_correct / val_total
    val_accuracy.append(val_acc)
    val_losses.append(val_loss)

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    precision = precision_score(all_labels, all_preds, average='macro', zero_division = 0)
    recall = recall_score(all_labels, all_preds, average='macro', zero_division = 0)
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division = 0)

    print(f"Epoch {epoch + 1}/{epochs}: Train Acc: {train_acc:.4f}, Train Loss: {train_loss:.4f}, Val Acc: {val_acc:.4f}, Val Loss: {val_loss:.4f}\n"
         f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1 score: {f1:.4f}")
print(classification_report(all_labels, all_preds, digits = 4))

Epoch 1/20: Train Acc: 0.4383, Train Loss: 1.2334, Val Acc: 0.5919, Val Loss: 1.0180
Precision: 0.5777, Recall: 0.5562, F1 score: 0.5548
Epoch 2/20: Train Acc: 0.6632, Train Loss: 0.8603, Val Acc: 0.7757, Val Loss: 0.6167
Precision: 0.7712, Recall: 0.7445, F1 score: 0.7496
Epoch 3/20: Train Acc: 0.7939, Train Loss: 0.5642, Val Acc: 0.8654, Val Loss: 0.3836
Precision: 0.8627, Recall: 0.8543, F1 score: 0.8567
Epoch 4/20: Train Acc: 0.8632, Train Loss: 0.3833, Val Acc: 0.9006, Val Loss: 0.2786
Precision: 0.8997, Recall: 0.8930, F1 score: 0.8954
Epoch 5/20: Train Acc: 0.8995, Train Loss: 0.2795, Val Acc: 0.9288, Val Loss: 0.1985
Precision: 0.9301, Recall: 0.9229, F1 score: 0.9260
Epoch 6/20: Train Acc: 0.9219, Train Loss: 0.2199, Val Acc: 0.9412, Val Loss: 0.1636
Precision: 0.9432, Recall: 0.9360, F1 score: 0.9391
Epoch 7/20: Train Acc: 0.9356, Train Loss: 0.1826, Val Acc: 0.9507, Val Loss: 0.1384
Precision: 0.9499, Recall: 0.9479, F1 score: 0.9488
Epoch 8/20: Train Acc: 0.9431, Train Loss